In [0]:
%run "../0_common/01. env-config"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:

from pyspark.sql import functions as F
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"
bronze_table, silver_table

In [0]:
circuits_df = spark.table(bronze_table).filter(F.col("batch_id")==v_batch_id)
display(circuits_df.head(5))

In [0]:
circuits_df = circuits_df.select("circuitId","circuitName","country","lat","long","locality","ingestion_timestamp","source_file","batch_id")

In [0]:
circuits_df = circuits_df.withColumnsRenamed(
    {
        "circuitId":"circuit_id",
        "circuitName":"circuit_name",
        "lat":"latitude",
        "long":"longitude"
    }
)

In [0]:
import pyspark.sql.functions as F
circuits_df = circuits_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
circuits_df = circuits_df.dropDuplicates(["circuit_id"])
display(circuits_df)

In [0]:
circuits_df = (
    circuits_df
    .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
    .withColumn("locality", F.initcap(F.col("locality")))
)

In [0]:
display(circuits_df.head(5))

In [0]:
circuits_df = (
    circuits_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
if not spark.catalog.tableExists(silver_table):
    circuits_df.write.format('delta').mode("overwrite").saveAsTable(silver_table)
else:
    from delta.tables import DeltaTable
    delta_table = DeltaTable.forName(spark, silver_table)
    delta_table.alias("t").merge(
        circuits_df.alias("s"),
        "t.circuit_id = s.circuit_id"
    ).whenMatchedUpdate(
        condition="s.batch_id >= t.batch_id",
        set={
            "circuit_name": "s.circuit_name",
            "latitude": "s.latitude",
            "longitude": "s.longitude",
            "locality": "s.locality",
            "country": "s.country",
            "ingestion_timestamp": "s.ingestion_timestamp",
            "source_file": "s.source_file",
            "batch_id": "s.batch_id",
            "updated_timestamp": "s.updated_timestamp" 
        }
    ).whenNotMatchedInsertAll().execute()

In [0]:
display(spark.table(silver_table))